In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


In [5]:
!pip install transformers --quiet

In [6]:

# ============================================================================
# CELL CONTENT BEGINS HERE — paste everything below into a Colab cell
# ============================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report, confusion_matrix,
)

# Install if needed (uncomment when running in Colab)
# !pip install transformers --quiet

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
INPUT_CSV = "/content/phase1_binary_corpus.csv"   # uploaded file
OUTPUT_DIR = "/content/phase1_distilbert"          # model save location
METRICS_PATH = "/content/phase1_metrics.json"

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 3                     # 3 is enough for binary on large corpus
LR = 2e-5
WEIGHT_DECAY = 0.01            # regularization to prevent overfitting
DROPOUT_PROB = 0.3             # higher than default 0.1 to fight memorization
WARMUP_RATIO = 0.1
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ----------------------------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------------------------
print("\n[1/5] Loading data...")
df = pd.read_csv(INPUT_CSV)
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)
df["text"] = df["text"].astype(str)
df = df[df["text"].str.strip() != ""].reset_index(drop=True)

print(f"  Total samples: {len(df)}")
print(f"  Scam: {(df['label']==1).sum()}, Legit: {(df['label']==0).sum()}")
print(f"  Sources: {df['source'].value_counts().to_dict()}" if 'source' in df.columns else "")

# Stratified 80/10/10 split
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
print(f"  Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

# ----------------------------------------------------------------------------
# DATASET CLASS
# ----------------------------------------------------------------------------
class ScamDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


# ----------------------------------------------------------------------------
# MODEL
# ----------------------------------------------------------------------------
print("\n[2/5] Loading DistilBERT...")
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dropout=DROPOUT_PROB,           # increased from default 0.1
    seq_classif_dropout=DROPOUT_PROB,
)
model.to(DEVICE)

train_ds = ScamDataset(train_df["text"], train_df["label"], tokenizer, MAX_LEN)
val_ds = ScamDataset(val_df["text"], val_df["label"], tokenizer, MAX_LEN)
test_ds = ScamDataset(test_df["text"], test_df["label"], tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

# ----------------------------------------------------------------------------
# TRAIN WITH EARLY STOPPING ON VAL F1
# ----------------------------------------------------------------------------
print("\n[3/5] Training...")
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)


def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=mask)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)


best_val_f1 = 0.0
best_epoch = 0
best_state = None
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    val_y, val_p = evaluate(val_loader)
    val_f1 = f1_score(val_y, val_p)
    train_y, train_p = evaluate(train_loader)
    train_f1 = f1_score(train_y, train_p)
    elapsed = (time.time() - t0) / 60

    print(f"  Epoch {epoch+1}/{EPOCHS} | loss={avg_loss:.4f} | "
          f"train F1={train_f1:.4f} | val F1={val_f1:.4f} | {elapsed:.1f}min")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch + 1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

# Restore best model
print(f"\n  Best epoch: {best_epoch} (val F1 = {best_val_f1:.4f})")
model.load_state_dict(best_state)

# ----------------------------------------------------------------------------
# FINAL EVALUATION ON TEST SET
# ----------------------------------------------------------------------------
print("\n[4/5] Evaluating on test set...")
train_y, train_p = evaluate(train_loader)
test_y, test_p = evaluate(test_loader)

metrics = {
    "phase": "phase1_binary_pretrain",
    "best_epoch": best_epoch,
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    "train_f1": float(f1_score(train_y, train_p)),
    "train_accuracy": float(accuracy_score(train_y, train_p)),
    "test_f1": float(f1_score(test_y, test_p)),
    "test_accuracy": float(accuracy_score(test_y, test_p)),
    "test_precision": float(precision_score(test_y, test_p, zero_division=0)),
    "test_recall": float(recall_score(test_y, test_p, zero_division=0)),
    "confusion_matrix": confusion_matrix(test_y, test_p).tolist(),
    "classification_report": classification_report(
        test_y, test_p, target_names=["legit", "scam"], digits=4, zero_division=0
    ),
}

print(f"\n  Train F1: {metrics['train_f1']:.4f}")
print(f"  Test  F1: {metrics['test_f1']:.4f}")
print(f"  Test Accuracy: {metrics['test_accuracy']:.4f}")
print(f"\n  Confusion matrix:\n{metrics['confusion_matrix']}")
print(f"\n  Classification report:\n{metrics['classification_report']}")

# ----------------------------------------------------------------------------
# SAVE MODEL
# ----------------------------------------------------------------------------
print("\n[5/5] Saving model...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\n  Saved model to: {OUTPUT_DIR}")
print(f"  Saved metrics to: {METRICS_PATH}")
print(f"\n  TOTAL TIME: {(time.time()-t0)/60:.1f} minutes")
print("\n" + "=" * 60)
print("PHASE 1 COMPLETE.")
print("Next step: Download the phase1_distilbert/ folder.")
print("Then run Phase 2 to fine-tune for tactic classification.")
print("=" * 60)

Using device: cuda

[1/5] Loading data...
  Total samples: 7433
  Scam: 2526, Legit: 4907
  Sources: {'emscad': 3364, 'uci_sms': 2747, 'your_synthetic': 1000, 'your_real': 322}
  Train: 5946  Val: 743  Test: 744

[2/5] Loading DistilBERT...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



[3/5] Training...
  Epoch 1/3 | loss=0.3991 | train F1=0.9020 | val F1=0.8685 | 1.8min
  Epoch 2/3 | loss=0.1895 | train F1=0.9367 | val F1=0.8961 | 3.6min
  Epoch 3/3 | loss=0.1380 | train F1=0.9473 | val F1=0.9175 | 5.2min

  Best epoch: 3 (val F1 = 0.9175)

[4/5] Evaluating on test set...

  Train F1: 0.9473
  Test  F1: 0.9065
  Test Accuracy: 0.9382

  Confusion matrix:
[[475, 16], [30, 223]]

  Classification report:
              precision    recall  f1-score   support

       legit     0.9406    0.9674    0.9538       491
        scam     0.9331    0.8814    0.9065       253

    accuracy                         0.9382       744
   macro avg     0.9368    0.9244    0.9302       744
weighted avg     0.9380    0.9382    0.9377       744


[5/5] Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Saved model to: /content/phase1_distilbert
  Saved metrics to: /content/phase1_metrics.json

  TOTAL TIME: 5.7 minutes

PHASE 1 COMPLETE.
Next step: Download the phase1_distilbert/ folder.
Then run Phase 2 to fine-tune for tactic classification.


In [9]:
!zip -r phase1_distilbert.zip /content/phase1_distilbert

  adding: content/phase1_distilbert/ (stored 0%)
  adding: content/phase1_distilbert/model.safetensors (deflated 8%)
  adding: content/phase1_distilbert/config.json (deflated 49%)
  adding: content/phase1_distilbert/tokenizer.json (deflated 71%)
  adding: content/phase1_distilbert/tokenizer_config.json (deflated 42%)


In [10]:
from google.colab import files
files.download("phase1_distilbert.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
"""
06_phase2_tactic_finetune.py
=============================
COLAB SCRIPT — Phase 2 multi-label tactic fine-tuning.

Loads the Phase 1 pre-trained model and specializes it for multi-label
psychological tactic classification across 4 categories:
  urgency, fomo, sunk_cost, social_proof

This is your PRIMARY research contribution.

HOW TO RUN ON GOOGLE COLAB:
  1. Make sure Phase 1 is done — you should have phase1_distilbert/ folder
  2. Upload these files to /content/:
        - phase1_distilbert/  (folder with model + tokenizer from Phase 1)
        - cleaned_synthetic.csv
        - augmented_real_filtered.csv
  3. Paste this entire file into ONE cell, run it.
  4. Wait ~10-15 minutes.
  5. Download phase2_distilbert/ folder.

EXPECTED RESULTS:
  Per-tactic F1: 0.70-0.88 (varies — sunk_cost weakest at ~0.65 due to fewer samples)
  Average tactic F1: 0.78-0.85
  These are realistic, defensible numbers — multi-label is harder than binary.

OUTPUT:
  /content/phase2_distilbert/   (final model — use this in FastAPI)
  /content/phase2_metrics.json  (per-tactic F1 scores)
"""

# ============================================================================
# PASTE EVERYTHING BELOW INTO ONE COLAB CELL
# ============================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    f1_score, precision_score, recall_score, hamming_loss,
    classification_report,
)

# !pip install transformers --quiet  # uncomment first time

from transformers import (
    DistilBertTokenizer,
    DistilBertModel,
    get_linear_schedule_with_warmup,
)

# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
PHASE1_MODEL_DIR = "/content/phase1_distilbert"   # output of script 05
SYN_CSV = "/content/cleaned_synthetic.csv"
REAL_CSV = "/content/augmented_real_filtered.csv"

OUTPUT_DIR = "/content/phase2_distilbert"
METRICS_PATH = "/content/phase2_metrics.json"

# 4 tactic labels — these MUST match the columns in your cleaned data
TACTIC_LABELS = ["urgency", "fomo", "sunk_cost", "social_proof"]
NUM_TACTICS = len(TACTIC_LABELS)

MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 8
LR = 2e-5
WEIGHT_DECAY = 0.01
DROPOUT_PROB = 0.3
WARMUP_RATIO = 0.1
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ----------------------------------------------------------------------------
# LOAD + PREPARE DATA
# ----------------------------------------------------------------------------
print("\n[1/6] Loading tactic-labeled data...")
syn = pd.read_csv(SYN_CSV)
real = pd.read_csv(REAL_CSV)

# Both files have the same schema thanks to script 01
needed_cols = ["msg_id", "text"] + [f"tactic_{t}" for t in TACTIC_LABELS]
for col in needed_cols:
    if col not in syn.columns:
        raise ValueError(f"Synthetic file missing column: {col}")
    if col not in real.columns:
        raise ValueError(f"Real file missing column: {col}")

syn["source"] = "synthetic"
real["source"] = "real"
syn["msg_id"] = syn["msg_id"].astype(str)
real["msg_id"] = real["msg_id"].astype(str)
# group_id from real (originals + their augmentations stay together)
real["group_id"] = real["msg_id"].str.split("_", n=1).str[0]
syn["group_id"] = "syn_" + syn["msg_id"]   # synthetic groups are themselves

cols_keep = ["msg_id", "text", "source", "group_id"] + [f"tactic_{t}" for t in TACTIC_LABELS]
df = pd.concat([syn[cols_keep], real[cols_keep]], ignore_index=True)
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)
df = df[df["text"].str.strip() != ""].reset_index(drop=True)

print(f"  Total: {len(df)}  (synthetic: {(df['source']=='synthetic').sum()}, "
      f"real: {(df['source']=='real').sum()})")
for t in TACTIC_LABELS:
    print(f"  {t}: {df[f'tactic_{t}'].sum()} positives")

# ----------------------------------------------------------------------------
# GROUP-AWARE SPLIT — originals + augmentations stay together
# ----------------------------------------------------------------------------
print("\n[2/6] Group-aware 80/10/10 split...")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
trainval_idx, test_idx = next(splitter.split(df, groups=df["group_id"]))
trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=SEED)
tr_idx, val_idx = next(splitter2.split(trainval_df, groups=trainval_df["group_id"]))
train_df = trainval_df.iloc[tr_idx].reset_index(drop=True)
val_df = trainval_df.iloc[val_idx].reset_index(drop=True)

print(f"  Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")

# ----------------------------------------------------------------------------
# DATASET — multi-label
# ----------------------------------------------------------------------------
class TacticDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df["text"].tolist()
        self.labels = df[[f"tactic_{t}" for t in TACTIC_LABELS]].values.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }


# ----------------------------------------------------------------------------
# MODEL — multi-label classifier head on top of Phase 1 DistilBERT
# ----------------------------------------------------------------------------
class TacticClassifier(nn.Module):
    def __init__(self, base_model, num_tactics, dropout=0.3):
        super().__init__()
        self.bert = base_model
        hidden = base_model.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_tactics)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # use CLS token representation
        cls = out.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls))
        loss = None
        if labels is not None:
            # binary cross-entropy with logits — handles multi-label
            loss = nn.BCEWithLogitsLoss()(logits, labels)
        return loss, logits


print("\n[3/6] Loading Phase 1 pre-trained backbone...")
tokenizer = DistilBertTokenizer.from_pretrained(PHASE1_MODEL_DIR)
# Load the encoder backbone from Phase 1 (drop its binary classification head)
base_model = DistilBertModel.from_pretrained(PHASE1_MODEL_DIR)
model = TacticClassifier(base_model, NUM_TACTICS, dropout=DROPOUT_PROB)
model.to(DEVICE)
print("  Phase 1 weights loaded successfully")

train_ds = TacticDataset(train_df, tokenizer, MAX_LEN)
val_ds = TacticDataset(val_df, tokenizer, MAX_LEN)
test_ds = TacticDataset(test_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

# ----------------------------------------------------------------------------
# TRAIN
# ----------------------------------------------------------------------------
print("\n[4/6] Training Phase 2...")
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)


def evaluate_multilabel(loader, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            _, logits = model(input_ids=input_ids, attention_mask=mask)
            all_logits.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    probs = np.vstack(all_logits)
    labels = np.vstack(all_labels)
    preds = (probs > threshold).astype(int)
    return labels, preds, probs


best_val_f1 = 0.0
best_state = None
best_epoch = 0
t0 = time.time()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        loss, _ = model(input_ids=input_ids, attention_mask=mask, labels=labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    val_y, val_p, _ = evaluate_multilabel(val_loader)
    val_f1_macro = f1_score(val_y, val_p, average="macro", zero_division=0)
    train_y, train_p, _ = evaluate_multilabel(train_loader)
    train_f1_macro = f1_score(train_y, train_p, average="macro", zero_division=0)
    elapsed = (time.time() - t0) / 60

    print(f"  Epoch {epoch+1}/{EPOCHS} | loss={avg_loss:.4f} | "
          f"train F1={train_f1_macro:.4f} | val F1={val_f1_macro:.4f} | {elapsed:.1f}min")

    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        best_epoch = epoch + 1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\n  Best epoch: {best_epoch} (val macro F1 = {best_val_f1:.4f})")
model.load_state_dict(best_state)

# ----------------------------------------------------------------------------
# FINAL EVALUATION
# ----------------------------------------------------------------------------
print("\n[5/6] Final evaluation on test set...")
train_y, train_p, _ = evaluate_multilabel(train_loader)
test_y, test_p, _ = evaluate_multilabel(test_loader)

per_tactic = {}
for i, t in enumerate(TACTIC_LABELS):
    per_tactic[t] = {
        "f1": float(f1_score(test_y[:, i], test_p[:, i], zero_division=0)),
        "precision": float(precision_score(test_y[:, i], test_p[:, i], zero_division=0)),
        "recall": float(recall_score(test_y[:, i], test_p[:, i], zero_division=0)),
        "support_test": int(test_y[:, i].sum()),
    }

metrics = {
    "phase": "phase2_tactic_finetune",
    "best_epoch": best_epoch,
    "n_train": len(train_df),
    "n_val": len(val_df),
    "n_test": len(test_df),
    "train_macro_f1": float(f1_score(train_y, train_p, average="macro", zero_division=0)),
    "test_macro_f1": float(f1_score(test_y, test_p, average="macro", zero_division=0)),
    "test_micro_f1": float(f1_score(test_y, test_p, average="micro", zero_division=0)),
    "test_hamming_loss": float(hamming_loss(test_y, test_p)),
    "per_tactic": per_tactic,
    "tactic_labels": TACTIC_LABELS,
}

print("\n  PER-TACTIC RESULTS (test set):")
print(f"  {'tactic':<15} {'F1':>7} {'P':>7} {'R':>7} {'support':>9}")
print("  " + "-" * 50)
for t in TACTIC_LABELS:
    s = per_tactic[t]
    print(f"  {t:<15} {s['f1']:>7.4f} {s['precision']:>7.4f} "
          f"{s['recall']:>7.4f} {s['support_test']:>9d}")

print(f"\n  Train macro F1: {metrics['train_macro_f1']:.4f}")
print(f"  Test  macro F1: {metrics['test_macro_f1']:.4f}")
print(f"  Test  micro F1: {metrics['test_micro_f1']:.4f}")
print(f"  Hamming loss:   {metrics['test_hamming_loss']:.4f}")

# ----------------------------------------------------------------------------
# SAVE MODEL
# ----------------------------------------------------------------------------
print("\n[6/6] Saving final model...")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save the full model state
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "model_state.pt"))

# Also save the BERT encoder backbone in HF format (for FastAPI loading)
model.bert.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save the classifier head separately
torch.save({
    "classifier_state": model.classifier.state_dict(),
    "num_tactics": NUM_TACTICS,
    "tactic_labels": TACTIC_LABELS,
    "dropout": DROPOUT_PROB,
}, os.path.join(OUTPUT_DIR, "tactic_head.pt"))

with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\n  Saved to: {OUTPUT_DIR}")
print(f"  Metrics : {METRICS_PATH}")
print(f"\n  TOTAL TIME: {(time.time()-t0)/60:.1f} minutes")

print("\n" + "=" * 60)
print("PHASE 2 COMPLETE.")
print("Download phase2_distilbert/ folder — this is your final model.")
print("Use it in your FastAPI backend for the demo.")
print("=" * 60)

Using device: cuda

[1/6] Loading tactic-labeled data...
  Total: 1322  (synthetic: 1000, real: 322)
  urgency: 419 positives
  fomo: 279 positives
  sunk_cost: 277 positives
  social_proof: 341 positives

[2/6] Group-aware 80/10/10 split...
  Train: 913  Val: 129  Test: 280

[3/6] Loading Phase 1 pre-trained backbone...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: /content/phase1_distilbert
Key                   | Status     |  | 
----------------------+------------+--+-
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Phase 1 weights loaded successfully

[4/6] Training Phase 2...
  Epoch 1/8 | loss=0.5490 | train F1=0.4485 | val F1=0.4135 | 0.3min
  Epoch 2/8 | loss=0.3913 | train F1=0.6462 | val F1=0.6079 | 0.6min
  Epoch 3/8 | loss=0.2991 | train F1=0.7867 | val F1=0.7150 | 0.9min
  Epoch 4/8 | loss=0.2372 | train F1=0.8165 | val F1=0.7020 | 1.1min
  Epoch 5/8 | loss=0.2138 | train F1=0.8483 | val F1=0.7265 | 1.4min
  Epoch 6/8 | loss=0.1931 | train F1=0.8836 | val F1=0.7308 | 1.7min
  Epoch 7/8 | loss=0.1678 | train F1=0.8903 | val F1=0.7127 | 1.9min
  Epoch 8/8 | loss=0.1583 | train F1=0.8987 | val F1=0.7196 | 2.2min

  Best epoch: 6 (val macro F1 = 0.7308)

[5/6] Final evaluation on test set...

  PER-TACTIC RESULTS (test set):
  tactic               F1       P       R   support
  --------------------------------------------------
  urgency          0.7500  0.9000  0.6429        98
  fomo             0.5263  0.7143  0.4167        72
  sunk_cost        0.8393  0.7705  0.9216        51
  social

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Saved to: /content/phase2_distilbert
  Metrics : /content/phase2_metrics.json

  TOTAL TIME: 2.5 minutes

PHASE 2 COMPLETE.
Download phase2_distilbert/ folder — this is your final model.
Use it in your FastAPI backend for the demo.


In [13]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

MODEL_PATH = "/content/phase1_distilbert"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_PATH)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()

def predict(text):
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt"
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)
        pred = torch.argmax(outputs.logits, dim=1).item()

    return "SCAM" if pred == 1 else "LEGIT"

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [16]:
texts = [
    "We are pleased to offer you a position. Please complete the background verification form attached.",
    "To finalize your onboarding, a refundable processing charge of $75 is required.",
    "Your resume has been shortlisted. HR will contact you within 2–3 business days.",
    "Due to high demand, we require a small commitment fee to proceed with your application.",
    "We are hiring a QA engineer with Selenium experience. Apply via our official portal.",
    "Kindly confirm your interest so we can schedule the final round interview.",
    "Secure your position today by completing the registration payment.",
    "Our company provides relocation assistance and health benefits.",
    "You have been selected for the next round. Please share your availability.",
    "Limited openings available. Immediate joining after verification payment.",
    "We are expanding our team and looking for experienced Java developers.",
    "Your offer letter will be released once the initial deposit is completed.",
    "Please attend the virtual interview session scheduled tomorrow.",
    "Earn passive income by completing online surveys in your free time.",
    "Your profile matches our requirements. Kindly wait for further updates.",
    "Fast-track your hiring process by paying the priority processing fee.",
    "We follow a structured interview process including technical and HR rounds.",
    "Start working immediately and earn daily payouts with no prior skills.",
    "The company offers annual bonuses and career growth opportunities.",
    "Submit your documents for verification to proceed with onboarding.",
]

for t in texts:
    print(t)
    print("Prediction:", predict(t))
    print("-"*50)

We are pleased to offer you a position. Please complete the background verification form attached.
Prediction: SCAM
--------------------------------------------------
To finalize your onboarding, a refundable processing charge of $75 is required.
Prediction: SCAM
--------------------------------------------------
Your resume has been shortlisted. HR will contact you within 2–3 business days.
Prediction: SCAM
--------------------------------------------------
Due to high demand, we require a small commitment fee to proceed with your application.
Prediction: SCAM
--------------------------------------------------
We are hiring a QA engineer with Selenium experience. Apply via our official portal.
Prediction: LEGIT
--------------------------------------------------
Kindly confirm your interest so we can schedule the final round interview.
Prediction: LEGIT
--------------------------------------------------
Secure your position today by completing the registration payment.
Prediction: SCAM

In [17]:
!zip -r phase2_distilbert.zip /content/phase2_distilbert

  adding: content/phase2_distilbert/ (stored 0%)
  adding: content/phase2_distilbert/model.safetensors (deflated 8%)
  adding: content/phase2_distilbert/model_state.pt (deflated 8%)
  adding: content/phase2_distilbert/config.json (deflated 48%)
  adding: content/phase2_distilbert/tokenizer.json (deflated 71%)
  adding: content/phase2_distilbert/tokenizer_config.json (deflated 43%)
  adding: content/phase2_distilbert/tactic_head.pt (deflated 14%)


In [18]:
from google.colab import files
files.download("phase2_distilbert.zip")
files.download("/content/phase2_metrics.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>